# Эксперимент 20: Признаки голосового тракта + SVM/MLP

**Статья:** Acoustic Predictors of Pediatric Dysarthria in Cerebral Palsy (Акустические предикторы педиатрической дизартрии при церебральном параличе) 2018

**Ссылка:** [https://pmc.ncbi.nlm.nih.gov/articles/PMC5963041/](https://pmc.ncbi.nlm.nih.gov/articles/PMC5963041/)

**Краткое описание модели:** Ручные акустические признаки (pitch, энергия, спектральные индексы, formant-подобные статистики) -> SVM/MLP.

**Содержание статьи:** Клинические исследования показывают, что нарушения артикуляции отражаются в устойчивых акустических индикаторах. Интерпретируемые признаки голосового тракта полезны как диагностический baseline и как референс к DL-подходам. Эксперимент оценивает их предсказательную силу на вашем датасете.

In [1]:
import sys
from pathlib import Path
import numpy as np
import time
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    precision_score,
    recall_score,
    classification_report,
)
import pandas as pd

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))

from shared import config, data_utils
from shared.results_utils import save_result_csv

## 1. Загрузка разбиений и извлечение признаков

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
print(f"Train: {len(paths_train)}, Val: {len(paths_val)}, Test: {len(paths_test)}")
print(f"Train: good={np.sum(y_train==0)}, bad={np.sum(y_train==1)}")

Train: 1942, Val: 417, Test: 417
Train: good=1311, bad=631


In [3]:
def extract(path):
    return data_utils.extract_vocal_tract_features(path)

X_train = data_utils.build_feature_matrix(paths_train, extract, n_jobs=-1)
X_val   = data_utils.build_feature_matrix(paths_val,   extract, n_jobs=-1)
X_test  = data_utils.build_feature_matrix(paths_test,  extract, n_jobs=-1)

X_train = np.hstack([X_train, letters_train])
X_val   = np.hstack([X_val, letters_val])
X_test  = np.hstack([X_test, letters_test])

print(f"Признаков на объект: {X_train.shape[1]}")

Признаков на объект: 32


## 2. Масштабирование, подбор гиперпараметров и обучение

In [4]:
classes = np.unique(y_train)
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes, weights))

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(kernel="rbf", class_weight=class_weight, probability=True, random_state=config.RANDOM_STATE)),
])
param_grid = {
    "clf__C": [0.1, 1.0, 10.0],
    "clf__gamma": ["scale", "auto"],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="f1_macro", refit=True, n_jobs=-1, verbose=1)
t0 = time.perf_counter()
grid.fit(X_train, y_train)
train_time_sec = time.perf_counter() - t0

print("Лучшие гиперпараметры:", grid.best_params_)
print("Лучший F1-macro (CV):", f"{grid.best_score_:.4f}")
clf = grid.best_estimator_

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Лучшие гиперпараметры: {'clf__C': 1.0, 'clf__gamma': 'scale'}
Лучший F1-macro (CV): 0.6215


## 3. Оценка на тесте и запись метрик

In [ ]:
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else None

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, average="binary", pos_label=1)
roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else None
precision_bad = precision_score(y_test, y_pred, zero_division=0, pos_label=1)
recall_bad = recall_score(y_test, y_pred, zero_division=0, pos_label=1)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 macro: {f1_macro:.4f}")
print(f"F1 (bad): {f1_bad:.4f}")
print(f"ROC-AUC:  {roc_auc}")

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_20_vocal_tract",
    experiment_name="Vocal tract + SVM",
    model="SVM (RBF)",
    accuracy=accuracy,
    f1_macro=f1_macro,
    f1_bad=f1_bad,
    roc_auc=roc_auc,
    precision_bad=precision_bad,
    recall_bad=recall_bad,
    notes="vocal tract (rms,zcr,F0,centroid,rolloff,bw + optional jitter/shimmer/formants)",
    num_params=None,
    train_time_sec=train_time_sec,
)

              precision    recall  f1-score   support

        good       0.78      0.68      0.73       282
         bad       0.47      0.60      0.53       135

    accuracy                           0.65       417
   macro avg       0.63      0.64      0.63       417
weighted avg       0.68      0.65      0.66       417

Accuracy: 0.6547
F1 macro: 0.6283
F1 (bad): 0.5294
ROC-AUC:  0.7005910165484633
Результат сохранён в result.csv текущего эксперимента
